In [1]:
from __future__ import annotations
from typing import Callable
import numpy as np
from numpy.random import default_rng

from mos.mos_simulator import MoSSimulator
from ql.prover import Prover, ProverConfig
from ql.verifier import Verifier, VerificationResult, VerifierConfig, ClassicalExampleOracle


In [2]:
def run_one_round(mos_simulator, phi_prob: Callable[[int], float],
                  prover_cfg: ProverConfig, verifier_cfg: VerifierConfig,
                  seed: int = 0) -> VerificationResult:
    rng = default_rng(seed)
    prover = Prover(mos_simulator, prover_cfg, rng=rng)
    oracle = ClassicalExampleOracle(mos_simulator.n, phi_prob, rng=rng)
    verifier = Verifier(mos_simulator.n, oracle, verifier_cfg)

    L = prover.propose_list()
    return verifier.verify(L)

In [3]:
from numpy.random import default_rng

# --- your phi_prob: Pr[b=1 | x] ---
def phi_prob(x: int) -> float:
    # example: strong parity signal on s = (2^n - 1)
    # (replace with your actual distribution)
    n = 3
    parity = (x.bit_count() & 1)  # parity(x)
    # b = parity(x) with 10% noise (so correlation is high)
    # P[b=1|x] = 0.1 if parity=0 else 0.9
    return 0.1 if parity == 0 else 0.9

# --- build your MoS simulator ---
n = 3
mos = MoSSimulator(n=n, phi=phi_prob, seed=123)

# --- parameter choices you requested ---
vartheta = 0.8
a = 0.4
b = 2
epsilon = 0.05

# Derived verifier parameters
max_list_size = 8
weight_threshold = (a * a) - (epsilon * epsilon) / 8.0  # 0.1596875
m_classical = 10_000

verifier_cfg = VerifierConfig(
    max_list_size=max_list_size,
    m_classical=m_classical,
    weight_threshold=weight_threshold,
    require_nonempty=True,
)

# Prover configuration: keep list small
prover_cfg = ProverConfig(
    shots=50_000,          # MoS samples; increase if list is unstable
    max_list_size=max_list_size,
    method="topk",
    topk=20,                # b=2 suggests at most 2 heavy terms
    use_oracle=True,
    mode="statevector",    # fast; use "circuit" for real backend
)

# --- run one interactive round ---
res = run_one_round(
    mos_simulator=mos,
    phi_prob=phi_prob,
    prover_cfg=prover_cfg,
    verifier_cfg=verifier_cfg,
    seed=0
)

print("accepted:", res.accepted)
print("reason:", res.reason)
print("weight:", res.weight)
print("s_out:", res.s_out)
print("L correlations:", res.xi_hat)


accepted: True
reason: accept
weight: 0.6496483200000001
s_out: 7
L correlations: {7: 0.8052, 3: -0.0094, 6: -0.027, 5: 0.0074, 4: 0.0128, 0: -0.0002, 2: -0.0068, 1: -0.0148}
